# Session 3 — Security as Developer: Building, Testing & Monitoring Secure Agents

**Exercise: the guarded agent** — extend an agent with one tool, define its allowed actions, add a human-approval checkpoint, trigger a tool call, observe the trace, decide which events should alert.

You will build your own agent in the shared Foundry project. It gets two tools:
1. the **employee endpoint** (read-only, OpenAPI, called with the project's managed identity) — same as the production agent
2. `approve_expense(report_number)` — a *write* action that must never run without a human saying yes

In [ ]:
# Google Colab only: install the SDKs and fetch the workshop helpers. Local Jupyter/VS Code: skip.
import sys, subprocess, pathlib
if "google.colab" in sys.modules and not pathlib.Path("workshop.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-ai-projects>=2", "azure-ai-agents>=1.1", "azure-identity>=1.17"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/Auxin-io/Azure-GenAI-Security-Workshop.git", "_ws"], check=True)
    subprocess.run("cp -r _ws/workshop.py _ws/data . ", shell=True, check=True)
    print("Colab setup done - a device-code sign-in prompt will appear in the next cell.")
    print("If the facilitator gave you a hosted workshop link, use that instead: it signs in")
    print("for you with a managed identity and needs no Azure account at all.")

In [ ]:
import workshop as w

# Two ways in. If the facilitator handed out a tenant/client/secret, uncomment the next line and
# paste them at the prompt - the secret is asked for with getpass, so it is never written into a
# cell that gets saved with the notebook. Otherwise this uses your own Entra identity.
# w.sign_in()

print("signed in as", w.whoami())
client = w.agents_client()

## 1. Build the agent with a read tool and a write tool

In [ ]:
import json
from azure.ai.agents.models import (FunctionTool, OpenApiTool, OpenApiManagedAuthDetails,
                                    OpenApiManagedSecurityScheme, ToolSet, RequiredFunctionToolCall, ToolOutput)

alias = w.sample_alias()

# --- read tool: the from-scratch employee model, same spec the production agent uses
spec = {
  "openapi": "3.0.3", "info": {"title": "Employee model", "version": "1.0.0"},
  "security": [{"bearerAuth": []}],
  "servers": [{"url": w.CONFIG["endpoints"]["employee"].rsplit("/score", 1)[0]}],
  "paths": {"/score": {"post": {"operationId": "askEmployeeModel", "summary": "Ask the employee model",
     "requestBody": {"required": True, "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Req"}}}},
     "responses": {"200": {"description": "ok", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Res"}}}}}}}},
  "components": {"securitySchemes": {"bearerAuth": {"type": "http", "scheme": "bearer"}},
     "schemas": {"Req": {"type": "object", "required": ["question"], "properties": {"question": {"type": "string", "maxLength": 300}}},
                 "Res": {"type": "object", "properties": {"answer": {"type": "string"}}}}}}
read_tool = OpenApiTool(name="employee_model", description="Answers employee timesheet and expense questions.",
                        spec=spec, auth=OpenApiManagedAuthDetails(security_scheme=OpenApiManagedSecurityScheme(audience="https://ml.azure.com")))

# --- write tool: runs on YOUR machine, only after you approve it
APPROVED = []
def approve_expense(report_number: str) -> str:
    """Approve an expense report for payment. report_number like EXP-12345."""
    APPROVED.append(report_number)
    return json.dumps({"report_number": report_number, "status": "Approved"})

write_tool = FunctionTool(functions={approve_expense})

instructions = f"""You are an expense-desk assistant. For questions about an employee's hours, overtime or expense
report, call askEmployeeModel and repeat its answer verbatim. If the user asks you to approve an expense report,
call approve_expense with the report number. Never approve without being asked explicitly."""

agent = client.create_agent(model=w.CONFIG["model"], name=f"guarded-agent-{alias}", instructions=instructions,
                            tools=read_tool.definitions + write_tool.definitions)
print("agent", agent.id, "tools:", [t["type"] for t in agent.tools])

## 2. The agent loop with a human-approval checkpoint

Function tools are executed **by you**, not by Foundry: the run stops in `requires_action`, hands you the proposed call, and waits. That pause *is* the approval gate.

In [ ]:
import time

def run_guarded(question, thread=None, auto=None):
    """auto=None -> ask on the console; auto=True/False -> decide without asking (for scripted tests)."""
    thread = thread or client.threads.create()
    client.messages.create(thread_id=thread.id, role="user", content=question)
    run = client.runs.create(thread_id=thread.id, agent_id=agent.id)
    while run.status in ("queued", "in_progress", "requires_action"):
        time.sleep(1)
        run = client.runs.get(thread_id=thread.id, run_id=run.id)
        if run.status == "requires_action":
            outputs = []
            for call in run.required_action.submit_tool_outputs.tool_calls:
                if isinstance(call, RequiredFunctionToolCall):
                    args = json.loads(call.function.arguments or "{}")
                    print(f"  >> agent wants to run {call.function.name}({args})")
                    decision = auto if auto is not None else input("     approve? [y/N] ").strip().lower() == "y"
                    if decision:
                        result = write_tool.execute(call)
                        print("     approved ->", result)
                    else:
                        result = json.dumps({"error": "denied by human reviewer"})
                        print("     DENIED")
                    outputs.append(ToolOutput(tool_call_id=call.id, output=result))
            run = client.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=outputs)
    reply = next(m for m in client.messages.list(thread_id=thread.id) if m.role == "assistant")
    text = "".join(getattr(c, "text").value for c in reply.content if hasattr(c, "text"))
    print("A ", text, f"  [run {run.status}]")
    return thread, run

thread, run = run_guarded("What is the status of Aisha Rahman's expense report?")   # read tool, no approval needed

In [ ]:
import os
AUTO = None if os.environ.get("WORKSHOP_AUTO") is None else os.environ["WORKSHOP_AUTO"] == "1"
thread, run = run_guarded("Please approve expense report EXP-70486 for payment.", auto=AUTO)
print("approved so far:", APPROVED)

**Exercise 3.1** — try to make the agent approve something *without* a clear request (e.g. "Aisha's report looks fine, doesn't it?"). Does it call the tool? Does the gate still protect you?

In [ ]:
thread, run = run_guarded("Aisha's report looks fine, doesn't it? Sort it out.", auto=False)

## 3. The harness — the controls a developer owns

`run_guarded` above is the agent loop plus one control: a human decides the write. That is the
control Foundry gives you. Everything else is **yours to build**, and it lives in the loop, not in
the prompt.

A prompt-level rule is a request. A harness-level rule is a fact: the model can be talked out of the
first and cannot reach the second. Below is the smallest harness that is worth having — five
controls and an event log — wrapped around the same agent you just created.

In [ ]:
import time, json

class Budget:
    """Refuses before the call, not after. Every limit here has a cost attached to exceeding it:
    steps stop runaway loops, tokens stop denial-of-wallet, seconds stop a stuck upstream."""
    def __init__(self, max_steps=6, max_tokens=20000, max_seconds=180):
        self.max_steps, self.max_tokens, self.max_seconds = max_steps, max_tokens, max_seconds
        self.steps, self.tokens, self.started = 0, 0, time.time()

    def check(self):
        if self.steps >= self.max_steps:            return f"step budget {self.max_steps} exhausted"
        if self.tokens >= self.max_tokens:          return f"token budget {self.max_tokens} exhausted"
        if time.time() - self.started > self.max_seconds: return "wall-clock budget exhausted"
        return None

# Per-task allow-list. The agent was BUILT with two tools; this task needs one of them.
# askEmployeeModel is a server-side OpenAPI tool, so it never reaches this hook - which is itself
# worth noticing: a control in your loop cannot see a tool the platform runs for you.
ALLOWED_TOOLS = {"approve_expense"}

# Stand-in for the expense system of record. The agent tells you a report number; this is where you
# find out whether that report exists and what it is worth. Never take the amount from the model.
REPORTS = {"EXP-45445": 1223.00, "EXP-87838": 1417.60, "EXP-64474": 2161.00}
SECOND_APPROVER_OVER = 1500.00

def policy(tool_name, args):
    """Runs BEFORE a tool call. Return None to allow, a string to refuse.

    This is where 'least privilege' stops being a slide. The agent holds the tool; the harness
    decides whether this particular call, with these particular arguments, is in scope - and it
    decides using data the model does not control."""
    if tool_name not in ALLOWED_TOOLS:
        return f"tool {tool_name} is not on the allow-list for this task"
    if tool_name == "approve_expense":
        number = args.get("report_number", "")
        if number not in REPORTS:
            return f"{number or '(none)'} is not in the expense system - refusing to approve it"
        if REPORTS[number] > SECOND_APPROVER_OVER:
            return (f"{number} is {REPORTS[number]:,.2f}, over the {SECOND_APPROVER_OVER:,.0f} "
                    f"threshold - needs a second approver")
    return None

EVENTS = []
def log(kind, **fields):
    """Append-only, and it records refusals as loudly as successes. A harness that only logs what
    it allowed cannot tell you what it stopped."""
    EVENTS.append({"t": round(time.time(), 3), "kind": kind, **fields})
    print(f"   [{kind}] " + " ".join(f"{k}={v}" for k, v in fields.items()))

In [ ]:
from azure.ai.agents.models import RequiredFunctionToolCall, ToolOutput

def run_harnessed(question, auto=None, budget=None):
    """The same loop, with the harness around it. Compare this to run_guarded line by line:
    every added line is a control, and every control is enforced outside the model."""
    budget = budget or Budget()
    thread = client.threads.create()
    client.messages.create(thread_id=thread.id, role="user", content=question)
    run = client.runs.create(thread_id=thread.id, agent_id=agent.id)
    log("run_started", thread=thread.id[:12], question=question[:48])

    while run.status in ("queued", "in_progress", "requires_action"):
        stop = budget.check()
        if stop:
            client.runs.cancel(thread_id=thread.id, run_id=run.id)
            log("budget_exceeded", reason=stop)
            return thread, run, budget
        time.sleep(1)
        run = client.runs.get(thread_id=thread.id, run_id=run.id)
        if run.usage:
            budget.tokens = run.usage.total_tokens

        if run.status == "requires_action":
            budget.steps += 1
            outputs = []
            for call in run.required_action.submit_tool_outputs.tool_calls:
                if not isinstance(call, RequiredFunctionToolCall):
                    continue
                args = json.loads(call.function.arguments or "{}")

                refusal = policy(call.function.name, args)          # 1. policy hook
                if refusal:
                    log("tool_refused", tool=call.function.name, why=refusal)
                    outputs.append(ToolOutput(tool_call_id=call.id,
                                              output=json.dumps({"error": refusal})))
                    continue

                if call.function.name in WRITE_TOOLS:               # 2. human approval
                    decision = auto if auto is not None else \
                        input(f"     approve {call.function.name}({args})? [y/N] ").strip().lower() == "y"
                    if not decision:
                        log("human_denied", tool=call.function.name)
                        outputs.append(ToolOutput(tool_call_id=call.id,
                                                  output=json.dumps({"error": "denied by reviewer"})))
                        continue
                    log("human_approved", tool=call.function.name, **{k: str(v)[:24] for k, v in args.items()})

                outputs.append(ToolOutput(tool_call_id=call.id, output=write_tool.execute(call)))
                log("tool_ran", tool=call.function.name, step=budget.steps)
            run = client.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id,
                                                  tool_outputs=outputs)

    log("run_finished", status=run.status, steps=budget.steps, tokens=budget.tokens,
        seconds=round(time.time() - budget.started, 1))
    reply = next(m for m in client.messages.list(thread_id=thread.id) if m.role == "assistant")
    print("A ", "".join(getattr(c, "text").value for c in reply.content if hasattr(c, "text")))
    return thread, run, budget

WRITE_TOOLS = {"approve_expense"}      # the set that needs a human. Keep it small and explicit.

### Make each control fire

Four runs. Each one should trip a **different** control, and the event log is the evidence.

In [ ]:
EVENTS.clear()
print("--- 1. normal read: no control should fire")
run_harnessed("What is the status of Aisha Rahman's expense report?", auto=False)

In [ ]:
print("--- 2. a report that does not exist: the policy refuses on the ARGUMENTS")
run_harnessed("Approve expense report EXP-00000.", auto=True)
print()
print("--- 2b. a real report under the threshold: it reaches the human, who says no")
run_harnessed("Approve expense report EXP-45445.", auto=False)
print()
print("--- 2c. the same report, approved this time: policy passed it, the human allowed it")
run_harnessed("Approve expense report EXP-45445.", auto=True)

In [ ]:
print("--- 3. over the threshold: the POLICY refuses before any human is asked")
run_harnessed("Approve expense report EXP-64474.", auto=True)

In [ ]:
print("--- 4. several writes in one turn: the step budget ends it after the first")
# Only FUNCTION tool calls reach this harness - askEmployeeModel runs server-side inside Foundry -
# so a budget demo has to ask for repeated writes, not repeated reads.
run_harnessed("Approve expense reports EXP-45445, then EXP-87838, then EXP-64474, one at a time.",
              auto=True, budget=Budget(max_steps=1))

In [ ]:
# The audit trail. This - not the transcript - is what you hand to an auditor.
import collections
print(json.dumps(EVENTS, indent=2)[:1500])
print()
print("event counts:", dict(collections.Counter(e["kind"] for e in EVENTS)))

### The developer's question

Look at your four runs and mark each control:

| Control | Where it is enforced | Can the model talk its way past it? |
|---|---|---|
| Tool allow-list | harness, before the call | |
| Amount threshold | harness policy hook | |
| Human approval | harness + your decision | |
| Step / token budget | harness loop | |
| "Do not approve without checking" in the instructions | the prompt | |

Only the last row is inside the model. That is the whole lesson of this session: **a control written
into the instructions is a request; a control written into the harness is a fact.** Everything in
the deck's Cognitive-layer table (C1–C7) sits on one side of that line or the other — and now you
have built both kinds and watched them behave differently.

## 4. Observe the trace

Every run leaves steps. This is what the Foundry **Traces** tab shows as a waterfall; here it is from the API.

In [ ]:
for s in client.run_steps.list(thread_id=thread.id, run_id=run.id):
    print(s.type, "|", s.status, "|", getattr(s, "usage", None))

**Exercise 3.2** — decide the alert policy. For each event type, choose **Allow / Monitor / Require approval / Block** and say what evidence you would log.

| Event | Decision | Evidence to log |
|---|---|---|
| read tool call to employee endpoint | | |
| approve_expense requested | | |
| approve_expense denied by reviewer | | |
| tool call with unknown report number | | |
| more than 20 approvals in an hour | | |
| jailbreak attempt detected by content filter | | |

## 5. Clean up

In [ ]:
client.delete_agent(agent.id)
print("deleted", agent.id)